# ProkBERT-mini shared promoter benchmark — Kaggle GPU

This notebook runs SeqTrainer's reusable ProkBERT benchmark on Kaggle. It uses the generic `neuralbioinfo/prokbert-mini` masked-language-model backbone, never the promoter-fine-tuned checkpoint, and never uses test data for tuning.

Contract: GSE144621, EP_DNA_BERT2_genomic_order, 300 bp sequences, predefined train/eval/test CSVs, seed 42, validation-MCC threshold selection, and held-out test evaluation after the best checkpoint is fixed. The model is CC-BY-NC-4.0. External ProkBERT scores are not SeqTrainer benchmark results.

## 0. Kaggle controls

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, shutil, subprocess, sys
import pandas as pd

SEQTRAINER_BRANCH = "issue-3-all-model-baselines"
SEQTRAINER_COMMIT = None  # Set to an exact SHA to pin a published run.
RUN_MODE = "full"         # "smoke" or "full"
RESUME_FROM_CHECKPOINT = True
KAGGLE_DATA_DIR = None    # Optional attached-dataset subdirectory; None searches /kaggle/input.

PROKBERT_MODEL = "neuralbioinfo/prokbert-mini"
HF_MODEL_REVISION = "feb2520a43cd9cdb5b3d8477e47209dbcb55d1dc"
PROKBERT_REPOSITORY = "https://github.com/nbrg-ppcu/prokbert.git"
PROKBERT_COMMIT = "8670ae92b816cff158a0b85647a8dea122e251eb"
REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"

REPO_DIR = Path("/kaggle/working/SeqTrainer")
LOCAL_RUN_ROOT = Path("/kaggle/working/prokbert_mini_kaggle_run")
OUTPUT_DIR = LOCAL_RUN_ROOT / ("smoke_test" if RUN_MODE == "smoke" else "full")
DATA_DIR = REPO_DIR / "data" / "promoter_classification"
CONFIG_PATH = REPO_DIR / "notebooks/final_training/config/prokbert_mini_kaggle.toml"
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
print({"run_mode": RUN_MODE, "output_dir": str(OUTPUT_DIR)})

## 1. Select and report the Kaggle GPU

In [ ]:
import torch
subprocess.run(["nvidia-smi"], check=False)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running this notebook.")
DEVICE = torch.device("cuda:0")
print("device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("torch:", torch.__version__)

## 2. Clone the correct SeqTrainer branch and pin the resolved commit

In [ ]:
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", SEQTRAINER_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", SEQTRAINER_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", SEQTRAINER_BRANCH, f"origin/{SEQTRAINER_BRANCH}"], check=True)
if SEQTRAINER_COMMIT:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", SEQTRAINER_COMMIT], check=True)
SEQTRAINER_COMMIT_RESOLVED = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
sys.path.insert(0, str(REPO_DIR / "src"))
required = [REPO_DIR / "src/seqtrainer/torch/prokbert_benchmark.py", CONFIG_PATH]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Checked-out branch is missing: " + ", ".join(missing))
print("SeqTrainer commit:", SEQTRAINER_COMMIT_RESOLVED)

## 3. Install Kaggle-compatible dependencies

Kaggle's CUDA-enabled PyTorch is retained. The official ProkBERT repository is installed explicitly at its pinned commit.

In [ ]:
def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *packages], check=True)
pip_install("transformers>=4.30", "datasets>=2.18", "accelerate>=0.26")
pip_install(f"git+{PROKBERT_REPOSITORY}@{PROKBERT_COMMIT}")
pip_install("--no-deps", "-e", str(REPO_DIR))

import datasets, transformers, prokbert
print("torch.__version__ =", torch.__version__)
print("transformers.__version__ =", transformers.__version__)
print("datasets.__version__ =", datasets.__version__)
print("ProkBERT module =", prokbert.__file__)
try:
    print("ProkBERT distribution =", importlib.metadata.version("prokbert"))
except importlib.metadata.PackageNotFoundError:
    print("ProkBERT distribution metadata unavailable; pinned commit =", PROKBERT_COMMIT)

## 4. Locate and validate the canonical dataset

Attach the Kaggle dataset containing the three unchanged CSVs. This discovers nested input directories and copies the files to writable local SeqTrainer storage.

In [ ]:
split_names = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}
search_root = Path(KAGGLE_DATA_DIR) if KAGGLE_DATA_DIR else Path("/kaggle/input")
if not search_root.exists():
    raise FileNotFoundError(f"Missing Kaggle input directory: {search_root}")
source_dir = None
for candidate in search_root.rglob(split_names["train"]):
    if all((candidate.parent / name).exists() for name in split_names.values()):
        source_dir = candidate.parent
        break
if source_dir is None:
    raise FileNotFoundError(
        "Attach a Kaggle dataset containing: " + ", ".join(split_names.values())
    )
DATA_DIR.mkdir(parents=True, exist_ok=True)
split_files = {}
split_frames = {}
split_summary = {}
for split, filename in split_names.items():
    destination = DATA_DIR / filename
    shutil.copy2(source_dir / filename, destination)
    split_files[split] = destination
    frame = pd.read_csv(destination)
    if not {"sequence", "label"}.issubset(frame.columns):
        raise ValueError(f"{destination} must contain sequence and label columns")
    if frame.empty or not set(frame["label"].dropna().unique()).issubset({0, 1}):
        raise ValueError(f"{destination} must be non-empty with binary labels 0/1")
    split_frames[split] = frame
    split_summary[split] = {
        "rows": len(frame),
        "label_counts": {str(k): int(v) for k, v in frame["label"].value_counts().sort_index().items()},
        "sha256": hashlib.sha256(destination.read_bytes()).hexdigest(),
        "sequence_length_min": int(frame["sequence"].astype(str).str.len().min()),
        "sequence_length_max": int(frame["sequence"].astype(str).str.len().max()),
    }
print("source:", source_dir)
print(json.dumps(split_summary, indent=2))

## 5. Load and assert the Kaggle configuration

In [ ]:
from seqtrainer.benchmarks import load_benchmark_config
config = load_benchmark_config(CONFIG_PATH)
assert config.model.family == "prokbert"
assert config.model.name == PROKBERT_MODEL
assert config.dataset.sequence_field == "sequence"
assert config.dataset.label_field == "label"
assert config.split.strategy == "predefined"
assert config.evaluation.primary_metric == "mcc"
assert config.model.params["revision"] == HF_MODEL_REVISION
assert config.model.params["tokenizer_revision"] == HF_MODEL_REVISION
print("config:", CONFIG_PATH)
print("HF revision:", config.model.params["revision"])

## 6. Load and audit the official ProkBERT tokenizer

In [ ]:
from seqtrainer.torch.prokbert_benchmark import audit_prokbert_tokenizer, load_prokbert_backbone
tokenizer, encoder = load_prokbert_backbone(
    PROKBERT_MODEL,
    revision=HF_MODEL_REVISION,
    tokenizer_revision=HF_MODEL_REVISION,
    trust_remote_code=True,
    local_files_only=False,
)
audit = audit_prokbert_tokenizer(
    tokenizer, encoder, split_frames["train"]["sequence"].head(5).tolist(), model_max_length=512
)
print(audit)
if audit.truncated_rows:
    raise RuntimeError(f"Tokenizer audit found {audit.truncated_rows} truncated rows.")

## 7. Device-correct forward-pass and memory preflight

The inputs and encoder are moved to the same CUDA device. The shared runner records any genuine OOM fallback while preserving effective batch size 32.

In [ ]:
encoder = encoder.to(DEVICE).eval()
sample = tokenizer(
    split_frames["train"]["sequence"].head(16).tolist(),
    padding="longest", truncation=True, max_length=512,
    return_attention_mask=True, return_tensors="pt",
)
sample = {key: value.to(DEVICE) for key, value in sample.items()}
with torch.no_grad():
    sample_out = encoder(input_ids=sample["input_ids"], attention_mask=sample["attention_mask"])
hidden = sample_out.last_hidden_state if hasattr(sample_out, "last_hidden_state") else sample_out[0]
print("input_ids:", tuple(sample["input_ids"].shape))
print("attention_mask:", tuple(sample["attention_mask"].shape))
print("encoder output:", tuple(hidden.shape))
print("allocated GiB:", round(torch.cuda.memory_allocated() / 2**30, 3))
del sample, sample_out, hidden
torch.cuda.empty_cache()
encoder = encoder.cpu()

## 8. Prepare full or smoke data and configure resumable output

In [ ]:
from dataclasses import replace
run_config = config
run_base = REPO_DIR
run_split_files = {split: str(path) for split, path in split_files.items()}

if RUN_MODE == "smoke":
    smoke_dir = LOCAL_RUN_ROOT / "smoke_data"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    smoke_paths = {}
    for split, frame in split_frames.items():
        parts = [group.sample(n=min(len(group), 16), random_state=42) for _, group in frame.groupby("label")]
        sampled = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=42)
        smoke_path = smoke_dir / f"{split}.csv"
        sampled.to_csv(smoke_path, index=False)
        smoke_paths[split] = str(smoke_path)
    run_config = replace(
        run_config,
        experiment=replace(run_config.experiment, name=f"{run_config.experiment.name}_smoke_test"),
        dataset=replace(run_config.dataset, split_files=smoke_paths),
        training=replace(run_config.training, max_epochs=1),
        outputs=replace(run_config.outputs, output_dir=str(OUTPUT_DIR)),
    )
    run_base = None
else:
    run_config = replace(
        run_config,
        dataset=replace(run_config.dataset, split_files=run_split_files),
        outputs=replace(run_config.outputs, output_dir=str(OUTPUT_DIR)),
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
resume_path = OUTPUT_DIR / "best_checkpoint.pt"
if RESUME_FROM_CHECKPOINT and resume_path.exists():
    run_config = replace(
        run_config,
        training=replace(
            run_config.training,
            params={**run_config.training.params, "resume_from_checkpoint": str(resume_path)},
        ),
    )
else:
    resume_path = None
print("mode:", RUN_MODE, "output:", OUTPUT_DIR, "resume:", resume_path)

## 9. Train through the shared SeqTrainer ProkBERT runner

In [ ]:
from seqtrainer.torch.prokbert_benchmark import run_prokbert_csv_splits
result = run_prokbert_csv_splits(
    run_config, base_dir=run_base, output_dir=OUTPUT_DIR,
    tokenizer=tokenizer, encoder=encoder,
)
print("status:", result.status, "output:", result.output_dir)
if result.status != "completed":
    raise RuntimeError(f"ProkBERT run did not complete: {result}")

## 10. Verify standard artifacts and inspect the report

In [ ]:
required_artifacts = [
    "metrics.csv", "metrics.json", "predictions.csv", "manifest.json",
    "history.csv", "best_checkpoint.pt", "tokenizer", "run_summary.txt",
]
missing_artifacts = [name for name in required_artifacts if not (OUTPUT_DIR / name).exists()]
if missing_artifacts:
    raise FileNotFoundError(f"Missing benchmark artifacts: {missing_artifacts}")
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text())
metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text())
predictions = pd.read_csv(OUTPUT_DIR / "predictions.csv")
print("verified:", required_artifacts)
print("prediction columns:", list(predictions.columns))
print("prediction rows:", len(predictions))
print("best validation MCC:", manifest.get("evaluation", {}).get("best_validation_mcc"))
print("selected validation threshold:", manifest.get("evaluation", {}).get("selected_validation_threshold"))
print("SMOKE TEST ONLY" if RUN_MODE == "smoke" else "FULL RUN")

## 11. Display learning curves and held-out confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
history = pd.read_csv(OUTPUT_DIR / "history.csv")
display(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train loss")
axes[0].plot(history["epoch"], history["validation_loss"], marker="o", label="validation loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].grid(alpha=0.25)
test_predictions = predictions[predictions["split"] == "test"]
cm = confusion_matrix(test_predictions["label"], test_predictions["prediction"], labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"]).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Held-out test confusion matrix")
plt.tight_layout(); plt.show()

## 12. Final Kaggle verification

Kaggle does not write to Google Drive. Use **Save Version → Save & Run All** so `/kaggle/working` artifacts are attached to the notebook output. Reruns with `RESUME_FROM_CHECKPOINT = True` restore only compatible checkpoints; SeqTrainer rejects mismatched configuration, split hashes, model revision, or commit.

Only a completed `RUN_MODE = "full"` execution on all three canonical files can be considered for `Results_Final.md`. Smoke output is diagnostic only.

In [ ]:
print("Final output directory:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path)
assert all((OUTPUT_DIR / name).exists() for name in required_artifacts)
print("All required ProkBERT artifacts are present.")
print("SeqTrainer commit:", SEQTRAINER_COMMIT_RESOLVED)
print("HF model revision:", HF_MODEL_REVISION)
print("ProkBERT repository commit:", PROKBERT_COMMIT)